In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml down minio

In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml up -d minio


#### Delete Existing Delta Table

In [2]:
%%bash

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/ --recursive


delete: s3://defaultfs/deltalake/delta_table/_delta_log/00000000000000000000.crc
delete: s3://defaultfs/deltalake/delta_table/_delta_log/00000000000000000000.json
delete: s3://defaultfs/deltalake/delta_table/_delta_log/_commits/
delete: s3://defaultfs/deltalake/delta_table/part-00000-777e89bb-de2c-4f42-ad2b-66e947c4945b-c000.snappy.parquet


In [3]:
%load_ext sql

In [4]:
%sql spark

#### Create Deltatable

In [12]:
%%sql

ALTER DATABASE default SET LOCATION 's3a://defaultfs/deltalake/'

Running query in 'SparkSession'

++
||
++
++

In [5]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

df = spark.read.format("csv").option("header", True).schema(schema).load("s3a://datasets/people-data/peoples.csv")
df.printSchema()

#
display(df)

root
 |-- id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- middleName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- ssn: string (nullable = true)
 |-- salary: integer (nullable = true)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30

#### Create Delta table

In [6]:
# Save as delta table into Minio S3
df.write.format('delta').save('/deltalake/delta_table')

# Create the table if it does not exist. Otherwise, replace the existing table.
#df.writeTo("spark_catalog.default.delta_table").createOrReplace()

# If you know the table does not already exist, you can call this instead:
#df.write.saveAsTable("spark_catalog.default.delta_table")

#### Read a Deltatable

In [7]:
delta_table = spark.read.format('delta').load("/deltalake/delta_table")
display(delta_table)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30:00|964-49-8051|79908 |
|6  |Chassidy  |Concepcion|Bourthouloume|F     |1990-11-24 10:30:00|954-59-9172|64652 |
|7  |Geri      |Tambra    |Mosby        |F     |1970-12-19 10:30:00|968-16-4020|38195 |
|8  |Patria    |Nancy     |Arstall      |F     |1985-01-02 10:30:00|984-76-3770|102053|
|9  |Terese    |Alfredia  |Tocqu

In [8]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/delta_table`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
0,2026-03-31 19:05:49,None,None,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}",None,None,None,None,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [9]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 's3a://defaultfs/deltalake/delta_table')
display(deltaTable.history())

+-------+-------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                       |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                               |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-03-31 19:05:49|NULL  |NULL    |WRITE    |{mode -> ErrorIfExists, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |true         |{numFi

In [ ]:
%%sql

INSERT INTO  delta.`s3a://defaultfs/deltalake/delta_table`
SELECT * FROM PARQUET.`s3a://defaultfs/deltalake/delta_table`;

In [10]:
%%sql

SELECT MIN(salary), MAX(salary), COUNT(id)
FROM delta.`s3a://defaultfs/deltalake/delta_table`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3
-6523,134393,1000


In [ ]:
%%sql 

UPDATE delta.`s3a://defaultfs/deltalake/delta_table`
SET salary = 6523
WHERE salary = -6523;

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/delta_table`

In [ ]:
%%sql 

SELECT *
FROM delta.`s3a://defaultfs/deltalake/delta_table`
WHERE id = 1

In [ ]:
%%sql 

DELETE
FROM delta.`s3a://defaultfs/deltalake/delta_table`
WHERE id = 1

In [ ]:
%%sql 

SELECT *
FROM delta.`s3a://defaultfs/deltalake/delta_table`
WHERE id = 1

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/delta_table`

In [ ]:
for i in range(5):
    spark.sql(
        """
        INSERT INTO delta.`s3a://defaultfs/deltalake/delta_table`
        SELECT * FROM PARQUET.`s3a://defaultfs/deltalake/delta_table`;
        """
    )
    print(f"{i}th INSERT completed ")

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/delta_table`

#### Python Create Deltatable

In [ ]:
spark.sql("""

CREATE OR REPLACE TABLE delta.`s3a://defaultfs/deltalake/delta_table` (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
) USING DELTA
          
""")

In [ ]:
from delta import *

DeltaTable.createIfNotExists(spark) \
    .tableName("people00m") \
    .addColumn("id", "INT") \
    .addColumn("firstName", "STRING") \
    .addColumn("middleName", "STRING") \
    .addColumn("lastName", "STRING", comment = "surname") \
    .addColumn("gender", "STRING") \
    .addColumn("birthDate", "TIMESTAMP") \
    .addColumn("ssn", "STRING") \
    .addColumn("salary", "INT") \
    .location("s3a://defaultfs/deltalake/delta_table") \
    .execute()

In [11]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, 's3a://defaultfs/deltalake/delta_table')
deltaTable.toDF().show()


+---+----------+----------+-------------+------+-------------------+-----------+------+
| id| firstName|middleName|     lastName|gender|          birthDate|        ssn|salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|  1|    Pennie|     Carry|   Hirschmann|     F|1955-07-02 09:30:00|981-43-9345| 56172|
|  2|        An|     Amira|       Cowper|     F|1992-02-08 10:30:00|978-97-8086| 40203|
|  3|     Quyen|    Marlen|         Dome|     F|1970-10-11 09:30:00|957-57-8246| 53417|
|  4|   Coralie|  Antonina|      Marshal|     F|1990-04-11 09:30:00|963-39-4885| 94727|
|  5|    Terrie|      Wava|        Bonar|     F|1980-01-16 10:30:00|964-49-8051| 79908|
|  6|  Chassidy|Concepcion|Bourthouloume|     F|1990-11-24 10:30:00|954-59-9172| 64652|
|  7|      Geri|    Tambra|        Mosby|     F|1970-12-19 10:30:00|968-16-4020| 38195|
|  8|    Patria|     Nancy|      Arstall|     F|1985-01-02 10:30:00|984-76-3770|102053|
|  9|    Terese|  Alfredia|     

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/delta_table`

In [35]:
%%sql 

CREATE TABLE IF NOT EXISTS spark_catalog.default.people12m (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
USING DELTA
--LOCATION 's3a://defaultfs/deltalake/people12m'

Running query in 'SparkSession'

++
||
++
++

In [38]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.default.people12m

Running query in 'SparkSession'

15 rows affected.

Field 1,Field 2,Field 3
id,int,None
firstName,string,None


In [44]:
%%sql 

DESCRIBE TABLE EXTENDED spark_catalog.default.people12m;

Running query in 'SparkSession'

15 rows affected.

Field 1,Field 2,Field 3
id,int,None
firstName,string,None


In [45]:
%%sql 

DESCRIBE HISTORY spark_catalog.default.people12m;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
0,2026-03-31 17:20:47,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [12]:
%%sql
-- In Spark SQL, the terms DATABASE and SCHEMA are interchangeable synonyms;
CREATE DATABASE IF NOT EXISTS deltalake
COMMENT 'This is a comment for the schema'
LOCATION 's3a://defaultfs/deltalake';

Running query in 'SparkSession'

++
||
++
++

In [27]:
%%sql

CREATE SCHEMA IF NOT EXISTS deltalake
COMMENT 'This is a comment for the schema'
LOCATION 's3a://defaultfs/deltalake';


Running query in 'SparkSession'

++
||
++
++

In [33]:
%%sql

USE deltalake;

Running query in 'SparkSession'

++
||
++
++

In [7]:
%%sql

ALTER DATABASE default SET LOCATION 's3a://defaultfs/deltalake/'

Running query in 'SparkSession'

++
||
++
++

In [48]:
# List tables in the current/default database
tables_list = spark.catalog.listTables()
print(tables_list)

# List tables in a specific database
tables_in_db = spark.catalog.listTables("default")
print(tables_in_db)

# Iterate and print table names (optional)
for table in tables_list:
    print(table.name)

[Table(name='people12m', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]
[Table(name='people12m', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]
people12m


#### Upsert to a Deltatable

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", DateType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

data = [
  (9999998, 'Billy', 'Tommie', 'Luppitt', 'M', date.fromisoformat('1992-09-17'), '953-38-9452', 55250),
  (9999999, 'Elias', 'Cyril', 'Leadbetter', 'M', date.fromisoformat('1984-05-22'), '906-51-2137', 48500),
  (10000000, 'Joshua', 'Chas', 'Broggio', 'M', date.fromisoformat('1968-07-22'), '988-61-6247', 90000),
  (20000001, 'John', '', 'Doe', 'M', date.fromisoformat('1978-01-14'), '345-67-8901', 55500),
  (20000002, 'Mary', '', 'Smith', 'F', date.fromisoformat('1982-10-29'), '456-78-9012', 98250),
  (20000003, 'Jane', '', 'Doe', 'F', date.fromisoformat('1981-06-25'), '567-89-0123', 89900)
]

delta_table_updates = spark.createDataFrame(data, schema)
delta_table_updates.createOrReplaceTempView("delta_table_updates")

# ...

from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, '/deltalake/delta_table')

(deltaTable.alias("delta_table")
  .merge(
    delta_table_updates.alias("delta_table_updates"),
    "delta_table.id = delta_table_updates.id"
  )
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)

#### Read a Deltatable

In [12]:
df = spark.read.format('delta').load("/deltalake/delta_table")
df_filtered = df.filter(df["id"] >= 9999998)
display(df_filtered)

+---+---------+----------+--------+------+---------+---+------+
|id |firstName|middleName|lastName|gender|birthDate|ssn|salary|
+---+---------+----------+--------+------+---------+---+------+
+---+---------+----------+--------+------+---------+---+------+



In [14]:
from pyspark.sql.functions import col
df = spark.read.option("versionAsOf", "0").table("delta.`/deltalake/delta_table`")
display(df.filter(col("id") == 1))

+---+---------+----------+----------+------+-------------------+-----------+------+
|id |firstName|middleName|lastName  |gender|birthDate          |ssn        |salary|
+---+---------+----------+----------+------+-------------------+-----------+------+
|1  |Pennie   |Carry     |Hirschmann|F     |1955-07-02 09:30:00|981-43-9345|56172 |
+---+---------+----------+----------+------+-------------------+-----------+------+



#### Write to a Deltatable

In [ ]:
# df.write.mode("append").saveAsTable("main.default.delta_table")

# Save as delta table
df.write.format('delta').mode('append').save('/deltalake/delta_table')

In [ ]:
# df.write.mode("overwrite").saveAsTable("main.default.delta_table")

# Save as delta table
df.write.format('delta').mode('overwrite').save('/deltalake/delta_table')

#### Update Deltatable Rows

In [41]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/delta_table')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "gender = 'F'",
  set = { "gender": "'Female'" }
)

# Declare the predicate by using Spark SQL functions.
deltaTable.update(
  condition = col('id') % lit(2) == 0,
  set = { "gender": "'Male'" }
)

In [43]:
df = spark.read.format('delta').load("/deltalake/delta_table")
# df.limit(20).show()

df_filtered = df.filter((col("gender") == "Male"))
display(df_filtered)

+---+---------+----------+-------------+------+-------------------+-----------+------+
|id |firstName|middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+---------+----------+-------------+------+-------------------+-----------+------+
|2  |An       |Amira     |Cowper       |Male  |1992-02-08 10:30:00|978-97-8086|40203 |
|4  |Coralie  |Antonina  |Marshal      |Male  |1990-04-11 09:30:00|963-39-4885|94727 |
|6  |Chassidy |Concepcion|Bourthouloume|Male  |1990-11-24 10:30:00|954-59-9172|64652 |
|8  |Patria   |Nancy     |Arstall      |Male  |1985-01-02 10:30:00|984-76-3770|102053|
|10 |Wava     |Lyndsey   |Jeandon      |Male  |1963-12-30 10:30:00|997-82-2946|56521 |
|12 |Jodie    |Tabetha   |Laneham      |Male  |1959-01-31 10:30:00|923-24-9769|90634 |
|14 |Caridad  |Maire     |Snelle       |Male  |1960-09-26 09:30:00|992-11-7062|38859 |
|16 |Chan     |Jani      |Hartas       |Male  |1986-12-05 10:30:00|995-51-3115|75050 |
|18 |Elnora   |Kecia     |Lipman       |Mal

#### Delete Rows 

In [44]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/delta_table')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("birthDate < '1955-01-01'")

# Declare the predicate by using Spark SQL functions.
deltaTable.delete(col('birthDate') < '1960-01-01')

#### Display table history

In [45]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/delta_table')
display(deltaTable.history())

+-------+-------------------+------+--------+---------+----------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                                       |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                                                                                                                                             

#### overwrite

In [ ]:
# Save as delta table
df.write.format('delta').mode('overwrite').save('/deltalake/delta_table')

#### Time Travle

In [46]:
# Read version 1
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/delta_table')
deltaHistory = deltaTable.history()

display(deltaHistory.where("version == 0"))
# Or:
#display(deltaHistory.where("timestamp == '2024-05-15T22:43:15.000+00:00'"))

+-------+-------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                       |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                               |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-03-31 19:05:49|NULL  |NULL    |WRITE    |{mode -> ErrorIfExists, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |true         |{numFi

In [47]:
df = spark.read.format('delta').option('versionAsOf', 0).load("/deltalake/delta_table")
# Or: 2025-10-14 18:39:41
#df = spark.read.format('delta').option('timestampAsOf', '2025-10-14T18:45:03.000+00:00').load("/deltalake/delta_table")

display(df)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30:00|964-49-8051|79908 |
|6  |Chassidy  |Concepcion|Bourthouloume|F     |1990-11-24 10:30:00|954-59-9172|64652 |
|7  |Geri      |Tambra    |Mosby        |F     |1970-12-19 10:30:00|968-16-4020|38195 |
|8  |Patria    |Nancy     |Arstall      |F     |1985-01-02 10:30:00|984-76-3770|102053|
|9  |Terese    |Alfredia  |Tocqu

#### Display table history

In [48]:
deltaTable = DeltaTable.forPath(spark, "/deltalake/delta_table")
print("######## Describe history for the table ######")
deltaTable.history().show()

######## Describe history for the table ######
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      5|2026-03-31 19:36:41|  NULL|    NULL|   DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          4|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/3.5....|
|      4|2026-03-31 19:36:39|  NULL|    NULL|   DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numRemovedFiles ...|   

#### Vacuum

In [49]:
deltaTable = DeltaTable.forPath(spark, "/deltalake/delta_table")
print("######## Vacuum the table ########")
deltaTable.vacuum()

######## Vacuum the table ########


Deleted 0 files and directories in a total of 1 directories.


DataFrame[]

#### Describe details for the table

In [50]:
print("######## Describe details for the table ######")
deltaTable.detail().show()

######## Describe details for the table ######
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|9fc83837-370f-469...|NULL|       NULL|s3a://defaultfs/d...|2026-03-31 19:05:...|2026-03-31 19:36:41|              []|               []|       1|      40191|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+

#### Generating manifest 

In [51]:
# Generate manifest
print("######## Generating manifest ######")
deltaTable.generate("SYMLINK_FORMAT_MANIFEST")

######## Generating manifest ######


In [52]:
# SQL Vacuum
print("####### SQL Vacuum #######")
spark.sql("VACUUM '%s' RETAIN 169 HOURS" % ("/deltalake/delta_table")).collect()

####### SQL Vacuum #######


Deleted 0 files and directories in a total of 1 directories.


[Row(path='s3a://defaultfs/deltalake/delta_table')]

In [53]:
# SQL describe history
print("####### SQL Describe History ########")
print(spark.sql("DESCRIBE HISTORY delta.`%s`" % ("/deltalake/delta_table")).collect())

####### SQL Describe History ########
[Row(version=5, timestamp=datetime.datetime(2026, 3, 31, 19, 36, 41), userId=None, userName=None, operation='DELETE', operationParameters={'predicate': '["(birthDate#10270 < 1960-01-01 00:00:00)"]'}, job=None, notebook=None, clusterId=None, readVersion=4, isolationLevel='Serializable', isBlindAppend=False, operationMetrics={'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '934', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '1', 'rewriteTimeMs': '133', 'numRemovedBytes': '44724', 'scanTimeMs': '801', 'numCopiedRows': '833', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '104', 'numAddedBytes': '40191'}, userMetadata=None, engineInfo='Apache-Spark/3.5.3 Delta-Lake/3.3.2'), Row(version=4, timestamp=datetime.datetime(2026, 3, 31, 19, 36, 39), userId=None, userName=None, operation='DELETE', operationParameters={'predicate': '["(birthDate#10270 < 1955-01-01 00:00:00)"]'}, job=None, notebook

In [ ]:
import shutil

# cleanup
shutil.rmtree("/tmp/delta_table")

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/delta_table --recursive

#### Optimize a table
After you have performed multiple changes to a table, you might have a lot of small files. To improve the speed of read queries, you can use the optimize operation to collapse small files into larger ones:

In [54]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.delta_table")
deltaTable = DeltaTable.forPath(spark, "/deltalake/delta_table")
deltaTable.optimize().executeCompaction()

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

#### Z-order by columns
To improve read performance further, you can collocate related information in the same set of files by z-ordering. Delta Lake data-skipping algorithms use this collocation to dramatically reduce the amount of data that needs to be read. To z-order data, you specify the columns to order on in the z-order by operation. For example, to collocate by gender, run:

In [55]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.delta_table")
deltaTable = DeltaTable.forPath(spark, "/deltalake/delta_table")
deltaTable.optimize().executeZOrderBy("gender")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

#### Clean up snapshots with VACUUM
Delta Lake provides snapshot isolation for reads, which means that it is safe to run an optimize operation even while other users or jobs are querying the table. Eventually however, you should clean up old snapshots. You can do this by running the vacuum operation:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.delta_table")
deltaTable = DeltaTable.forPath(spark, "/deltalake/delta_table")
deltaTable.vacuum()

#### How do I find the last commit's version in the Spark session?

In [ ]:
spark.conf.get("spark.databricks.delta.lastCommitVersionInSession")